
# Media Content Exposure Pilot — 실제 데이터 병행 산출 및 교차검증

`docs/MEDIA_CONTENT_EXPOSURE_PILOT.md`가 설명하듯, "미디어·콘텐츠 노출 지수(Media/Content
Exposure Index)"는 `docs/METHODOLOGY.md`의 한 줄 정의(예능/유튜브/영화/드라마 4종 서브태그
키워드 매칭) 외에 이번 세션에 코드나 JSON이 전혀 남아있지 않다 — Ad/Commercial·Fandom
Cohesion Pilot보다도 적은 흔적이다.

대신 두 가지 실측 데이터를 병행 활용한다.

1. **LDA 메타요인 F5 "미디어노출형(방송·조회수)"** — `fandom_scores_v6.json`의 `factor_share`
   (키워드: 예능·출연·드라마·지역·에서·프로그램·출연해·참여).
2. **v7 10·11라운드 병합 로그** — `v7_rounds/merge_log_r10.json`/`merge_log_r11.json`.
   이 라운드들의 설명 자체가 "미디어 크로스오버 리서치 — 예능 출연/드라마 출연/영화 배역
   출연/유튜브 활동 4개 각도"라고 명시하고 있어, 원본 지수가 채점하려 했던 원재료가 실제로
   코퍼스에 수집됐다는 직접적인 증거다.

이 노트북은 ①을 원본의 병행 대체 지표로 검증·순위화하고, ②를 이용해 그 대체 지표가
실제 "미디어 크로스오버 리서치"로 늘어난 근거문장과 얼마나 일치하는지(또는 하지 않는지)
정직하게 교차검증한다.


In [1]:

import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/v6_r22_snapshot")
FACTOR = "미디어노출형(방송·조회수)"

with open(DATA_DIR / "fandom_scores_v6.json", encoding="utf-8") as f:
    scores = json.load(f)
with open(DATA_DIR / "v7_rounds" / "merge_log_r10.json", encoding="utf-8") as f:
    r10 = json.load(f)
with open(DATA_DIR / "v7_rounds" / "merge_log_r11.json", encoding="utf-8") as f:
    r11 = json.load(f)

print("r10 round id:", r10["round"], "| net_new_bullets:", r10["net_new_bullets"], "| n_touched_fandoms:", r10["n_touched_fandoms"])
print("r11 round id:", r11["round"], "| net_new_bullets:", r11["net_new_bullets"], "| n_touched_fandoms:", r11["n_touched_fandoms"])
print(f"fandom_scores_v6.json: {len(scores)}개 팬덤")


r10 round id: v7_r10_media_crossover_research | net_new_bullets: 131 | n_touched_fandoms: 79
r11 round id: v7_r11_media_crossover_reinforcement | net_new_bullets: 143 | n_touched_fandoms: 68
fandom_scores_v6.json: 100개 팬덤



## 1. 데이터 무결성 검증

`factor_share`가 6개 메타요인에 대해 합 1.0이 되는지, `dominant_factor`가 실제로
`factor_share`의 최댓값과 일치하는지 100개 팬덤 전원에 대해 재확인한다(이전 파일럿들과
동일한 검증).


In [2]:

sum_mismatch = []
dominant_mismatch = []

for d in scores:
    fname = d["fandom"]
    shares = d["factor_share"]
    s = sum(shares.values())
    if abs(s - 1.0) > 0.001:
        sum_mismatch.append((fname, round(s, 4)))

    recomputed_dominant = max(shares, key=shares.get)
    if recomputed_dominant != d["dominant_factor"]:
        dominant_mismatch.append((fname, recomputed_dominant, d["dominant_factor"]))

print(f"factor_share 합 != 1.0 인 팬덤 수: {len(sum_mismatch)} / {len(scores)}")
print(f"dominant_factor 재계산 불일치 팬덤 수: {len(dominant_mismatch)} / {len(scores)}")


factor_share 합 != 1.0 인 팬덤 수: 0 / 100
dominant_factor 재계산 불일치 팬덤 수: 0 / 100



## 2. "미디어노출형(방송·조회수)" 비중 — 원본 지수에 대응하는 병행 지표

100개 팬덤 전체를 F5 비중 내림차순으로 정렬한다.


In [3]:

rows = []
for d in scores:
    rows.append({
        "팬덤": d["fandom"],
        "구분": d["category"],
        "근거문장수(activity)": d["activity"],
        "미디어노출형 비중": round(d["factor_share"].get(FACTOR, 0.0), 4),
        "대표 메타요인(6개 중)": d["dominant_factor"],
        "대표 메타요인이 미디어노출형인가": d["dominant_factor"] == FACTOR,
    })

media_df = pd.DataFrame(rows).sort_values("미디어노출형 비중", ascending=False).reset_index(drop=True)
media_df.index = media_df.index + 1
media_df.head(15)


,팬덤,구분,근거문장수(activity),미디어노출형 비중,대표 메타요인(6개 중),대표 메타요인이 미디어노출형인가
1,엔믹스,K-pop 걸그룹,71,0.1596,현장경제형(콘서트·투어·매진),False
2,BABYMONSTER,K-pop 걸그룹,80,0.1581,현장경제형(콘서트·투어·매진),False
3,장윤정,트로트,41,0.1488,현장경제형(콘서트·투어·매진),False
4,윤도현,록,30,0.1479,현장경제형(콘서트·투어·매진),False
5,거미,발라드,27,0.1452,현장경제형(콘서트·투어·매진),False
6,BTS,K-pop 보이그룹,125,0.1439,현장경제형(콘서트·투어·매진),False
7,김연자,트로트,47,0.1381,현장경제형(콘서트·투어·매진),False
8,동방신기,K-pop 보이그룹,54,0.1368,현장경제형(콘서트·투어·매진),False
9,츄,솔로,56,0.1336,현장경제형(콘서트·투어·매진),False
10,오마이걸,K-pop 걸그룹,53,0.1321,현장경제형(콘서트·투어·매진),False


In [4]:

n_dominant = media_df["대표 메타요인이 미디어노출형인가"].sum()
print(f"대표 메타요인이 '미디어노출형(방송·조회수)'인 팬덤: {n_dominant} / {len(media_df)}")
print(media_df[media_df["대표 메타요인이 미디어노출형인가"]][["팬덤", "구분", "미디어노출형 비중"]].to_string(index=False))


대표 메타요인이 '미디어노출형(방송·조회수)'인 팬덤: 0 / 100
Empty DataFrame
Columns: [팬덤, 구분, 미디어노출형 비중]
Index: []



## 3. 실제 코퍼스 증거와의 교차검증 — v7 10·11라운드 "미디어 크로스오버 리서치"

r10·r11 두 라운드의 `per_group_added`(그 라운드에서 팬덤별로 실제 새로 추가된 근거문장 수)를
합산해, F5 비중과 실제로 얼마나 상관관계가 있는지 계산한다. 이는 "이름이 비슷한 지표이니
당연히 일치할 것"이라 가정하지 않고 직접 검증하기 위함이다.


In [5]:

combined_new = {}
for log in (r10, r11):
    for fandom, n in log["per_group_added"].items():
        combined_new[fandom] = combined_new.get(fandom, 0) + n

media_df["r10+r11 신규 근거문장 수"] = media_df["팬덤"].map(combined_new).fillna(0).astype(int)

print(f"r10+r11 신규 근거문장 총합: {sum(combined_new.values())}건 "
      f"(r10 {r10['net_new_bullets']}건 + r11 {r11['net_new_bullets']}건 = {r10['net_new_bullets']+r11['net_new_bullets']}건과 일치 여부: "
      f"{sum(combined_new.values()) == r10['net_new_bullets']+r11['net_new_bullets']})")

corr = np.corrcoef(media_df["r10+r11 신규 근거문장 수"], media_df["미디어노출형 비중"])[0, 1]
print(f"\nPearson r(미디어노출형 비중, r10+r11 신규 근거문장 수) = {corr:.4f}")
print("-> 상관관계가 약하다는 것은 두 값이 서로 다른 것을 측정하기 때문으로 해석된다:")
print("   F5 비중=코퍼스 전체(5,612건) 대비 상대비율, r10+r11 신규건수=두 라운드에서만 늘어난 절대건수.")


r10+r11 신규 근거문장 총합: 274건 (r10 131건 + r11 143건 = 274건과 일치 여부: True)

Pearson r(미디어노출형 비중, r10+r11 신규 근거문장 수) = 0.1891
-> 상관관계가 약하다는 것은 두 값이 서로 다른 것을 측정하기 때문으로 해석된다:
   F5 비중=코퍼스 전체(5,612건) 대비 상대비율, r10+r11 신규건수=두 라운드에서만 늘어난 절대건수.



## 4. 신규 근거문장을 가장 많이 받은 팬덤 vs F5 비중이 가장 높은 팬덤

두 순위가 실제로 얼마나 겹치는지(또는 겹치지 않는지) 상위 10개씩 나란히 비교한다.


In [6]:

top_by_new = media_df.sort_values("r10+r11 신규 근거문장 수", ascending=False).head(10)[
    ["팬덤", "구분", "r10+r11 신규 근거문장 수", "미디어노출형 비중"]
].reset_index(drop=True)
top_by_share = media_df.sort_values("미디어노출형 비중", ascending=False).head(10)[
    ["팬덤", "구분", "미디어노출형 비중", "r10+r11 신규 근거문장 수"]
].reset_index(drop=True)

print("=== r10+r11 신규 근거문장 수 상위 10 ===")
print(top_by_new.to_string(index=False))
print()
print("=== 미디어노출형(F5) 비중 상위 10 ===")
print(top_by_share.to_string(index=False))

overlap = set(top_by_new["팬덤"]) & set(top_by_share["팬덤"])
print(f"\n두 상위 10 목록의 교집합: {len(overlap)}개 -> {sorted(overlap) if overlap else '없음'}")


=== r10+r11 신규 근거문장 수 상위 10 ===
             팬덤            구분  r10+r11 신규 근거문장 수  미디어노출형 비중
      BLACKPINK     K-pop 걸그룹                  7     0.0931
            로이킴           발라드                  7     0.1297
             지코            힙합                  7     0.0834
지드래곤 (G-Dragon)            힙합                  7     0.0816
            임영웅           트로트                  7     0.0670
            이찬원           트로트                  6     0.0909
             싸이            솔로                  5     0.0675
           10CM            솔로                  5     0.0723
           DAY6 K-pop 보이그룹 /록                  5     0.0669
            권은비            솔로                  5     0.0602

=== 미디어노출형(F5) 비중 상위 10 ===
         팬덤         구분  미디어노출형 비중  r10+r11 신규 근거문장 수
        엔믹스  K-pop 걸그룹     0.1596                  2
BABYMONSTER  K-pop 걸그룹     0.1581                  5
        장윤정        트로트     0.1488                  4
        윤도현          록     0.1479                  1
         거미   


## 5. 한계 (문서에서 이미 밝힌 것 재확인)

1. **채점된 원본 지수는 이번 세션에 존재한 적이 없다** — 이 노트북의 모든 수치는 ①LDA
   메타요인 F5 비중, ②v7 10·11라운드 병합 로그라는 두 개의 서로 다른 실측 데이터를
   병행·교차 활용한 것이지, "미디어·콘텐츠 노출 지수"라는 이름의 산출물을 복구하거나
   재현한 것이 아니다.
2. **r10+r11 신규 근거문장 수는 "그 라운드에 새로 추가된 것"만 센 값**이며, 그 이전
   라운드에 이미 존재했을 수 있는 미디어 노출 관련 문장(예: 초기 코퍼스에 이미 있던 예능
   출연 언급)은 포함하지 않는다 — "팬덤별 미디어 노출 총량"이 아니다.
3. F5 비중과 r10+r11 신규 근거문장 수의 상관관계가 약하게 나온 것(3절)은 오류가 아니라
   두 지표가 측정하는 대상 자체가 다르기 때문이며, 이를 인위적으로 맞추거나 감추지 않고
   그대로 보고했다.
4. 코퍼스 규모·계산 방법이 원본과 다르므로, 이 노트북의 표는 round22 기준의 **독립적인
   병행 산출물**이지 원본 Media/Content Exposure Index의 재현이 아니다.
